# 🧠 Semantic Caching

**Cache similar queries, not just exact matches**

---

## 📋 Overview

**What you'll learn:**
- Semantic similarity caching
- Embedding-based lookup
- Similarity thresholds
- Vector store integration
- Cost optimization

**Time estimate:** ⏱️ 55 minutes | **Difficulty:** 🔴 Advanced

---

In [ ]:
from openai import OpenAI
import os
import numpy as np
from typing import Optional, List, Tuple
import time

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 Why Semantic Caching?

### Traditional Caching Problem:

```python
# Only exact matches hit cache
cache.get("What is machine learning?")     # ❌ MISS
cache.get("What is machine learning?")     # ✅ HIT (exact)
cache.get("What's machine learning?")      # ❌ MISS (different!)
cache.get("Define machine learning")       # ❌ MISS (different!)
cache.get("Explain machine learning")      # ❌ MISS (different!)

Problem: Same question, different wording = cache miss
```

### Semantic Caching Solution:

```python
# Similar questions hit cache
semantic_cache.get("What is machine learning?")     # ❌ MISS
semantic_cache.get("What is machine learning?")     # ✅ HIT (exact)
semantic_cache.get("What's machine learning?")      # ✅ HIT (similar!)
semantic_cache.get("Define machine learning")       # ✅ HIT (similar!)
semantic_cache.get("Explain machine learning")      # ✅ HIT (similar!)

Solution: Use embeddings to find semantically similar queries
```

### How It Works:

```
1. User query → Embed → [0.1, 0.5, -0.3, ...]
                         ↓
2. Compare with cached embeddings
                         ↓
3. Find most similar (cosine similarity)
                         ↓
4. If similarity > threshold → Return cached response
   Else → Call API and cache
```

## 🔢 Basic Semantic Cache

In [ ]:
import numpy as np
from typing import List, Tuple, Optional

def cosine_similarity(a: List[float], b: List[float]) -> float:
    """Calculate cosine similarity between two vectors."""
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

class SemanticCache:
    """Cache that matches semantically similar queries."""
    
    def __init__(self, similarity_threshold: float = 0.95):
        """
        Args:
            similarity_threshold: Minimum similarity to consider a hit (0-1)
        """
        self.similarity_threshold = similarity_threshold
        self.cache: List[Tuple[str, List[float], str]] = []  # (query, embedding, response)
        self.hits = 0
        self.misses = 0
    
    def _get_embedding(self, text: str) -> List[float]:
        """Get embedding for text."""
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding
    
    def get(self, query: str) -> Optional[Tuple[str, float]]:
        """Get cached response for similar query.
        
        Returns:
            (response, similarity) if found, else None
        """
        if not self.cache:
            self.misses += 1
            return None
        
        # Get query embedding
        query_embedding = self._get_embedding(query)
        
        # Find most similar cached query
        best_match = None
        best_similarity = -1.0
        
        for cached_query, cached_embedding, response in self.cache:
            similarity = cosine_similarity(query_embedding, cached_embedding)
            
            if similarity > best_similarity:
                best_similarity = similarity
                best_match = (cached_query, response)
        
        # Check if similarity meets threshold
        if best_similarity >= self.similarity_threshold:
            self.hits += 1
            print(f"  ✅ Semantic HIT (similarity: {best_similarity:.3f})")
            print(f"     Matched: '{best_match[0]}'")
            return best_match[1], best_similarity
        
        self.misses += 1
        print(f"  ❌ Semantic MISS (best: {best_similarity:.3f})")
        return None
    
    def set(self, query: str, response: str):
        """Store query and response with embedding."""
        query_embedding = self._get_embedding(query)
        self.cache.append((query, query_embedding, response))
        print(f"  💾 Cached: '{query}'")
    
    def get_stats(self) -> dict:
        """Get cache statistics."""
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total > 0 else 0
        
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": f"{hit_rate:.1f}%",
            "size": len(self.cache),
            "threshold": self.similarity_threshold
        }

# Example usage
print("\n🧠 Semantic Cache Example\n")

cache = SemanticCache(similarity_threshold=0.95)

# First query
print("Query 1: 'What is machine learning?'")
result = cache.get("What is machine learning?")
if not result:
    cache.set("What is machine learning?", "ML is a subset of AI...")

# Similar queries (should hit cache)
print("\nQuery 2: 'What's machine learning?'")
result = cache.get("What's machine learning?")

print("\nQuery 3: 'Define machine learning'")
result = cache.get("Define machine learning")

print("\nCache stats:")
for key, value in cache.get_stats().items():
    print(f"  {key}: {value}")

## 🎯 Similarity Threshold Tuning

In [ ]:
print("""
# Similarity Threshold Guide

Threshold Trade-offs:

## High Threshold (0.98-1.0):
✅ Very accurate matches
✅ Low false positives
❌ Low hit rate
❌ Misses paraphrases

Examples:
  Query: "What is AI?"
  Match: "What is AI?" ✅ (0.99)
  Match: "What's AI?" ❌ (0.97)

## Medium Threshold (0.90-0.95):
✅ Good balance
✅ Catches paraphrases
✅ Reasonable accuracy
⚠️  Some false positives

Examples:
  Query: "What is AI?"
  Match: "What is AI?" ✅ (0.99)
  Match: "What's AI?" ✅ (0.97)
  Match: "Define AI" ✅ (0.93)
  Match: "Explain AI" ✅ (0.91)

## Low Threshold (0.80-0.90):
✅ High hit rate
✅ Catches variations
❌ More false positives
❌ May match unrelated queries

Examples:
  Query: "What is AI?"
  Match: "How does AI work?" ⚠️ (0.85) - Different!
  Match: "AI definition" ✅ (0.88)

## Recommendations:

Use Case              | Threshold | Reason
----------------------|-----------|--------
FAQ/Support           | 0.92-0.95 | Common variations expected
Exact definitions     | 0.95-0.98 | Need precision
General chat          | 0.90-0.93 | Balance cost/accuracy
Expensive operations  | 0.88-0.92 | Maximize cache hits
Medical/Legal         | 0.97-0.99 | Accuracy critical

💡 Start with 0.95 and adjust based on metrics!
""")

## 🚀 Production Semantic Cache with ChromaDB

In [ ]:
print("""
# Production semantic caching with ChromaDB

import chromadb
from openai import OpenAI
from typing import Optional, Tuple
import time

class ProductionSemanticCache:
    \"\"\"Production-ready semantic cache with ChromaDB.\"\"\" 
    
    def __init__(
        self,
        similarity_threshold: float = 0.95,
        collection_name: str = "llm_cache"
    ):
        self.client = OpenAI()
        self.chroma_client = chromadb.Client()
        self.similarity_threshold = similarity_threshold
        
        # Create or get collection
        self.collection = self.chroma_client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
    
    def _get_embedding(self, text: str):
        \"\"\"Get embedding from OpenAI.\"\"\" 
        response = self.client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return response.data[0].embedding
    
    def get(
        self,
        query: str,
        model: str = "gpt-3.5-turbo"
    ) -> Optional[Tuple[str, float]]:
        \"\"\"Get cached response for similar query.\"\"\" 
        
        # Get query embedding
        query_embedding = self._get_embedding(query)
        
        # Search for similar queries
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=1,
            where={"model": model}
        )
        
        if not results['ids'][0]:
            return None
        
        # Get similarity (distance in chromadb)
        distance = results['distances'][0][0]
        similarity = 1 - distance  # Convert distance to similarity
        
        if similarity >= self.similarity_threshold:
            response = results['metadatas'][0][0]['response']
            matched_query = results['documents'][0][0]
            
            return response, similarity, matched_query
        
        return None
    
    def set(
        self,
        query: str,
        response: str,
        model: str = "gpt-3.5-turbo",
        metadata: dict = None
    ):
        \"\"\"Store query and response.\"\"\" 
        
        query_embedding = self._get_embedding(query)
        
        # Prepare metadata
        meta = {
            "response": response,
            "model": model,
            "timestamp": time.time()
        }
        if metadata:
            meta.update(metadata)
        
        # Add to collection
        self.collection.add(
            embeddings=[query_embedding],
            documents=[query],
            metadatas=[meta],
            ids=[f"{hash(query)}_{time.time()}"]  # Unique ID
        )
    
    def clear(self):
        \"\"\"Clear all cached entries.\"\"\" 
        self.chroma_client.delete_collection(self.collection.name)
        self.collection = self.chroma_client.create_collection(
            name=self.collection.name
        )

# Usage
cache = ProductionSemanticCache(similarity_threshold=0.95)

def get_completion_cached(prompt: str, model: str = "gpt-3.5-turbo"):
    \"\"\"Get LLM completion with semantic caching.\"\"\" 
    
    # Check semantic cache
    result = cache.get(prompt, model)
    
    if result:
        response, similarity, matched = result
        print(f"✅ Semantic cache hit (similarity: {similarity:.3f})")
        print(f"   Matched: {matched}")
        return response
    
    # Cache miss - call API
    print("❌ Cache miss - calling API")
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    
    result = response.choices[0].message.content
    
    # Cache for future
    cache.set(prompt, result, model)
    
    return result

# Test
print(get_completion_cached("What is machine learning?"))
print(get_completion_cached("What's ML?"))  # Should hit cache!

✅ Benefits:
  - Vector search with HNSW
  - Fast similarity lookup
  - Persistent storage
  - Production-ready
""")

## 💰 Cost Savings Analysis

In [ ]:
class CostAnalyzer:
    """Analyze cost savings from semantic caching."""
    
    def __init__(self):
        self.exact_hits = 0
        self.semantic_hits = 0
        self.misses = 0
        
        # Pricing (per 1k tokens)
        self.gpt35_cost = 0.0015
        self.gpt4_cost = 0.03
        self.embedding_cost = 0.0001
        
        self.avg_tokens_per_request = 500
    
    def record_exact_hit(self):
        """Record exact cache hit (no embedding needed)."""
        self.exact_hits += 1
    
    def record_semantic_hit(self):
        """Record semantic cache hit (embedding needed)."""
        self.semantic_hits += 1
    
    def record_miss(self):
        """Record cache miss."""
        self.misses += 1
    
    def get_report(self, model: str = "gpt-3.5-turbo") -> str:
        """Generate cost analysis report."""
        
        total = self.exact_hits + self.semantic_hits + self.misses
        
        # Calculate costs
        llm_cost_per_request = (
            (self.avg_tokens_per_request / 1000) * 
            (self.gpt4_cost if model == "gpt-4" else self.gpt35_cost)
        )
        
        embedding_cost_per_request = (
            (100 / 1000) *  # ~100 tokens for query
            self.embedding_cost
        )
        
        # Without caching
        cost_without_cache = total * llm_cost_per_request
        
        # With semantic caching
        cost_with_cache = (
            # Exact hits: free
            0 +
            # Semantic hits: just embedding cost
            (self.semantic_hits * embedding_cost_per_request) +
            # Misses: embedding + LLM
            (self.misses * (embedding_cost_per_request + llm_cost_per_request))
        )
        
        savings = cost_without_cache - cost_with_cache
        savings_pct = (savings / cost_without_cache * 100) if cost_without_cache > 0 else 0
        
        report = f"""
💰 Cost Savings Report
{'='*60}

Model: {model}
Total requests: {total:,}

Cache Performance:
  Exact hits: {self.exact_hits:,} ({self.exact_hits/max(total,1)*100:.1f}%)
  Semantic hits: {self.semantic_hits:,} ({self.semantic_hits/max(total,1)*100:.1f}%)
  Misses: {self.misses:,} ({self.misses/max(total,1)*100:.1f}%)
  Overall hit rate: {(self.exact_hits + self.semantic_hits)/max(total,1)*100:.1f}%

Cost Analysis:
  Without caching: ${cost_without_cache:.4f}
  With semantic caching: ${cost_with_cache:.4f}
  
  💵 Savings: ${savings:.4f} ({savings_pct:.1f}%)

Per-Request Costs:
  LLM call: ${llm_cost_per_request:.6f}
  Embedding: ${embedding_cost_per_request:.6f}
  
Semantic Cache Value:
  Semantic hits saved: ${self.semantic_hits * llm_cost_per_request:.4f}
  Embedding overhead: ${(self.semantic_hits + self.misses) * embedding_cost_per_request:.4f}
  Net semantic benefit: ${(self.semantic_hits * llm_cost_per_request) - ((self.semantic_hits + self.misses) * embedding_cost_per_request):.4f}
        """
        
        return report

# Example analysis
analyzer = CostAnalyzer()

# Simulate traffic
for _ in range(30):
    analyzer.record_exact_hit()  # Exact matches

for _ in range(50):
    analyzer.record_semantic_hit()  # Similar queries

for _ in range(20):
    analyzer.record_miss()  # New queries

print(analyzer.get_report(model="gpt-3.5-turbo"))
print("\n" + "="*60 + "\n")
print(analyzer.get_report(model="gpt-4"))

## ✅ Summary

### Semantic Caching vs Traditional:

**Traditional Cache:**
```python
# Only exact matches
cache.get("What is AI?")      # ✅ HIT
cache.get("What's AI?")       # ❌ MISS (different)

Hit rate: ~20-30% (only exact matches)
```

**Semantic Cache:**
```python
# Similar queries hit too
cache.get("What is AI?")      # ✅ HIT
cache.get("What's AI?")       # ✅ HIT (similar!)
cache.get("Define AI")        # ✅ HIT (similar!)

Hit rate: ~60-80% (exact + similar)
```

### Implementation:

**1. Basic (In-Memory)**
```python
cache = SemanticCache(similarity_threshold=0.95)

# Check cache
result = cache.get(query)
if not result:
    result = call_api(query)
    cache.set(query, result)

✅ Good for: Small datasets, development
```

**2. Production (ChromaDB)**
```python
cache = ProductionSemanticCache(similarity_threshold=0.95)

# Handles embeddings and vector search
result = cache.get(query)

✅ Good for: Production, large scale
```

### Similarity Thresholds:

```python
# High precision (0.95-0.98)
cache = SemanticCache(similarity_threshold=0.96)
# Only very similar queries match

# Balanced (0.90-0.95)
cache = SemanticCache(similarity_threshold=0.93)
# Good balance of hits and accuracy

# High recall (0.85-0.90)
cache = SemanticCache(similarity_threshold=0.88)
# More hits but some false positives
```

### Cost Analysis:

**Costs per request:**
```
LLM call (GPT-3.5): $0.00075
LLM call (GPT-4): $0.015
Embedding: $0.00001
```

**Savings example (100k requests):**
```
Traditional caching (30% hit rate):
  70,000 LLM calls × $0.00075 = $52.50

Semantic caching (70% hit rate):
  30,000 LLM calls × $0.00075 = $22.50
  100,000 embeddings × $0.00001 = $1.00
  Total: $23.50

Savings: $29/day = $870/month
```

### Best Practices:

**1. Choose Right Threshold**
```python
# Start with 0.95, then tune
cache = SemanticCache(similarity_threshold=0.95)

# Monitor false positives
if false_positive_rate > 0.05:
    increase_threshold()
```

**2. Combine with Traditional Cache**
```python
# Try exact match first (fast, free)
result = traditional_cache.get(query)

# Then try semantic (slower, costs embedding)
if not result:
    result = semantic_cache.get(query)
```

**3. Monitor Performance**
```python
# Track metrics
metrics = {
    'exact_hits': 30,
    'semantic_hits': 50,
    'misses': 20,
    'false_positives': 2
}
```

**4. Use Appropriate Vector Store**
```python
# Small scale: In-memory
# Medium scale: ChromaDB
# Large scale: Pinecone, Weaviate, Qdrant
```

### When to Use:

**✅ Use semantic caching when:**
- Users ask similar questions differently
- FAQ/support chatbots
- High traffic with paraphrasing
- Cost optimization is important

**❌ Don't use when:**
- Queries are always unique
- Need exact precision (medical, legal)
- Very low traffic
- Embedding cost > LLM savings

### Congratulations! 🎉

You now understand advanced caching strategies!

**Next topics:**
- Database integration
- Observability
- Production deployment

### Next module: `10_databases/`